In [1]:
import os
import re
import json
import time
import requests
import pandas as pd
from openpyxl.styles import PatternFill, Font, Alignment, Border, Side
from openpyxl.utils import get_column_letter

BASE_URL    = "http://localhost:3000"
SCRIPT_NAME = "contatos-e-relacionados"

LOGIN_EMAIL    = "mathews0912.camargo@hotmail.com"
LOGIN_PASSWORD = "Teste@123"

_auth = requests.post(
    f"{BASE_URL}/auth/login",
    json={"email": LOGIN_EMAIL, "password": LOGIN_PASSWORD},
    headers={"Content-Type": "application/json"},
)
_auth.raise_for_status()

HEADERS = {
    "Authorization": f"Bearer {_auth.json()['token']}",
    "Content-Type": "application/json",
}

print("Autenticado com sucesso.")

Autenticado com sucesso.


In [2]:
import psycopg2
import psycopg2.extras

DB_CONFIG = {
    "host":            "10.210.10.15",
    "port":            5432,
    "user":            "postgres",
    "password":        "sofkronoos@@2020",
    "dbname":          "kronoos-bases",
    "connect_timeout": 10,
}

_db_conn = psycopg2.connect(**DB_CONFIG)
_db_conn.autocommit = True
print("Conexão com o banco estabelecida.")

Conexão com o banco estabelecida.


In [3]:
input_path = os.path.join("..", "input", "contatos-e-relacionados_documentos.json")

with open(input_path, "r", encoding="utf-8") as f:
    document_list = json.load(f)

print(f"Total de documentos a processar: {len(document_list)}")

Total de documentos a processar: 73


In [4]:
CACHE_DIR = os.path.join("..", "responses", SCRIPT_NAME, "cache")
os.makedirs(CACHE_DIR, exist_ok=True)


def normalize_doc(document):
    return re.sub(r"\D", "", document)


def get_main_phone(phones):
    if not phones:
        return "N/D"
    candidates = [p for p in phones if p.get("main")]
    if not candidates:
        candidates = phones
    best = max(candidates, key=lambda p: p.get("rate", 0))
    phone = str(best.get("phone", "")).strip()
    return phone if phone else "N/D"


def get_main_email(emails):
    if not emails:
        return "N/D"
    candidates = [e for e in emails if e.get("main")]
    if not candidates:
        candidates = emails
    best = max(candidates, key=lambda e: e.get("rate", 0))
    email = str(best.get("email", "")).strip()
    return email if email else "N/D"


_SQL_PESSOA = """
    SELECT ddd1, fone1, email1
    FROM pessoa_fisica
    WHERE cpf = %s
    LIMIT 1
"""


def fetch_pessoa_fisica(cpf):
    if not cpf or cpf == "N/D":
        return "N/D", "N/D"
    try:
        with _db_conn.cursor(cursor_factory=psycopg2.extras.RealDictCursor) as cur:
            cur.execute(_SQL_PESSOA, (cpf,))
            row = cur.fetchone()
        if not row:
            return "N/D", "N/D"
        ddd   = str(row["ddd1"]   or "").strip()
        fone  = str(row["fone1"]  or "").strip()
        email = str(row["email1"] or "").strip()
        telefone = f"({ddd}) {fone}" if ddd and fone else (fone or "N/D")
        return telefone, (email or "N/D")
    except Exception as e:
        print(f"  [DB] Erro ao buscar CPF {cpf}: {e}")
        return "N/D", "N/D"


THIN_BORDER = Border(
    left=Side(style="thin", color="D0D0D0"),
    right=Side(style="thin", color="D0D0D0"),
    top=Side(style="thin", color="D0D0D0"),
    bottom=Side(style="thin", color="D0D0D0"),
)


def format_sheet(ws, df, col_colors):
    header_align = Alignment(horizontal="center", vertical="center", wrap_text=True)
    data_align   = Alignment(horizontal="left",   vertical="center", wrap_text=False)

    for col_idx, col_name in enumerate(df.columns, start=1):
        header_hex, data_hex = col_colors.get(col_name, ("D9D9D9", "F5F5F5"))
        header_fill = PatternFill("solid", fgColor=header_hex)
        data_fill   = PatternFill("solid", fgColor=data_hex)

        header_cell = ws.cell(row=1, column=col_idx)
        header_cell.fill      = header_fill
        header_cell.font      = Font(bold=True, color="3B3B3B", size=10)
        header_cell.alignment = header_align
        header_cell.border    = THIN_BORDER

        for row_idx in range(2, ws.max_row + 1):
            cell           = ws.cell(row=row_idx, column=col_idx)
            cell.fill      = data_fill
            cell.border    = THIN_BORDER
            cell.font      = Font(size=10)
            cell.alignment = data_align

    ws.row_dimensions[1].height = 32

    for col_idx, col_name in enumerate(df.columns, start=1):
        col_letter = get_column_letter(col_idx)
        max_content = max(
            len(str(col_name)),
            max(
                (len(str(ws.cell(row=r, column=col_idx).value or "")) for r in range(2, ws.max_row + 1)),
                default=0,
            ),
        )
        ws.column_dimensions[col_letter].width = min(max_content + 3, 55)

    ws.freeze_panes = "A2"


print("Funções auxiliares carregadas. Cache em:", CACHE_DIR)

Funções auxiliares carregadas. Cache em: ../responses/contatos-e-relacionados/cache


In [5]:
WEBHOOK     = "https://webhook.site/c1a70033-31b9-4879-a4e2-6f7ffbb7bd40"
COST_CENTER = 1


def fetch_dados_gerais(document, max_retries=15, wait_sec=2):
    doc_key    = normalize_doc(document)
    cache_file = os.path.join(CACHE_DIR, f"{doc_key}.json")

    if os.path.exists(cache_file):
        print(f"  -> Cache encontrado, pulando chamada à API")
        with open(cache_file, "r", encoding="utf-8") as f:
            return json.load(f)

    body = {
        "document":    doc_key,
        "cost_center": COST_CENTER,
        "webhook_url": WEBHOOK,
    }
    try:
        r = requests.post(f"{BASE_URL}/dados-gerais", headers=HEADERS, json=body)
        r.raise_for_status()
        order_id = r.json()["order_id"]
    except Exception as e:
        print(f"  Erro ao criar ordem [{document}]: {e}")
        return None

    for _ in range(max_retries):
        try:
            r = requests.get(f"{BASE_URL}/dados-gerais/{order_id}", headers=HEADERS)
            r.raise_for_status()
            result = r.json()
            if result.get("status") == "SUCESSO":
                with open(cache_file, "w", encoding="utf-8") as f:
                    json.dump(result, f, ensure_ascii=False, indent=2)
                return result
        except Exception as e:
            print(f"  Erro ao buscar ordem {order_id} [{document}]: {e}")
            return None
        time.sleep(wait_sec)

    print(f"  Timeout aguardando resultado para {document} (order_id={order_id})")
    return None


print("Função de consulta carregada.")

Função de consulta carregada.


In [6]:
def extract_rows(documento, payload):
    """
    Retorna (com_relacionados, sem_relacionado).
    com_relacionados: lista de dicts, uma por pessoa em pessoas_contato.
    sem_relacionado:  dict único se não houver pessoas_contato, senão None.
    """
    data = payload.get("data", {})
    dp   = data.get("dados_pessoais", {})

    nome_pesquisado = dp.get("name", "N/D")
    telefone        = get_main_phone(dp.get("phones", []))
    email           = get_main_email(dp.get("emails", []))

    pessoas_contato = data.get("pessoas_contato", [])  # nível data, não dados_pessoais

    if not pessoas_contato:
        sem = {
            "Nome":      nome_pesquisado,
            "Documento": documento,
            "Telefone":  telefone,
            "E-mail":    email,
        }
        return [], sem

    com = []
    for pessoa in pessoas_contato:
        cpf_rel          = pessoa.get("document", "N/D")
        tel_rel, email_rel = fetch_pessoa_fisica(cpf_rel)
        com.append({
            "Nome":                   nome_pesquisado,
            "Documento":              documento,
            "Telefone":               telefone,
            "E-mail":                 email,
            "Nome Relacionado":       pessoa.get("name", "N/D"),
            "Documento Relacionado":  cpf_rel,
            "Tipo de Relação":        pessoa.get("relation", "N/D"),
            "Telefone Relacionado":   tel_rel,
            "E-mail Relacionado":     email_rel,
        })

    return com, None


print("Função de extração carregada.")

Função de extração carregada.


In [7]:
com_relacionados = []
sem_relacionados = []
errors           = []

for entry in document_list:
    documento = entry.get("documento", "")
    print(f"\nProcessando: {documento}")

    payload = fetch_dados_gerais(documento)
    if payload is None:
        errors.append({"Documento": documento})
        continue

    com_rows, sem_row = extract_rows(documento, payload)
    com_relacionados.extend(com_rows)
    if sem_row:
        sem_relacionados.append(sem_row)

output_dir = os.path.join("..", "responses", SCRIPT_NAME)
os.makedirs(output_dir, exist_ok=True)

print(f"\n{'='*50}")
print(f"RESUMO DA EXECUÇÃO")
print(f"{'='*50}")
print(f"Total de documentos buscados    : {len(document_list)}")
print(f"Com relacionados ativos         : {len(set(r['Documento'] for r in com_relacionados))}")
print(f"Sem relacionados ativos         : {len(sem_relacionados)}")
print(f"Erros                           : {len(errors)}")


Processando: 47137479200
  -> Cache encontrado, pulando chamada à API

Processando: 28002695615
  -> Cache encontrado, pulando chamada à API

Processando: 12036463649
  -> Cache encontrado, pulando chamada à API

Processando: 35168765653
  -> Cache encontrado, pulando chamada à API

Processando: 07611951653
  -> Cache encontrado, pulando chamada à API

Processando: 01431501620
  -> Cache encontrado, pulando chamada à API

Processando: 16133196653
  -> Cache encontrado, pulando chamada à API

Processando: 19945701991
  -> Cache encontrado, pulando chamada à API

Processando: 00241636434
  -> Cache encontrado, pulando chamada à API

Processando: 04758480478
  -> Cache encontrado, pulando chamada à API

Processando: 09431047691
  -> Cache encontrado, pulando chamada à API

Processando: 30381720659
  -> Cache encontrado, pulando chamada à API

Processando: 03665976634
  -> Cache encontrado, pulando chamada à API

Processando: 39383059672
  -> Cache encontrado, pulando chamada à API

Proce

In [8]:
COLS_COM = [
    "Nome",
    "Documento",
    "Telefone",
    "E-mail",
    "Nome Relacionado",
    "Documento Relacionado",
    "Tipo de Relação",
    "Telefone Relacionado",
    "E-mail Relacionado",
]

COLS_SEM = [
    "Nome",
    "Documento",
    "Telefone",
    "E-mail",
]

# Amarelo    → identidade pesquisada (Nome, Documento)
# Azul-claro → contato do pesquisado (Telefone, E-mail)
# Roxo       → relacionado (Nome, Documento, Tipo)
# Verde      → contato do relacionado via banco (Telefone Relacionado, E-mail Relacionado)
COLORS_COM = {
    "Nome":                   ("FFD966", "FFFCE8"),
    "Documento":              ("FFD966", "FFFCE8"),
    "Telefone":               ("BDD7EE", "F0F6FC"),
    "E-mail":                 ("BDD7EE", "F0F6FC"),
    "Nome Relacionado":       ("C9A0DC", "F5EEF8"),
    "Documento Relacionado":  ("C9A0DC", "F5EEF8"),
    "Tipo de Relação":        ("C9A0DC", "F5EEF8"),
    "Telefone Relacionado":   ("A9D18E", "EEF5E9"),
    "E-mail Relacionado":     ("A9D18E", "EEF5E9"),
}

COLORS_SEM = {
    "Nome":      ("FFD966", "FFFCE8"),
    "Documento": ("FFD966", "FFFCE8"),
    "Telefone":  ("BDD7EE", "F0F6FC"),
    "E-mail":    ("BDD7EE", "F0F6FC"),
}

df_com = pd.DataFrame(com_relacionados, columns=COLS_COM) if com_relacionados else pd.DataFrame(columns=COLS_COM)
df_sem = pd.DataFrame(sem_relacionados, columns=COLS_SEM) if sem_relacionados else pd.DataFrame(columns=COLS_SEM)

output_file = os.path.join(output_dir, "relatorio_contatos_e_relacionados.xlsx")

with pd.ExcelWriter(output_file, engine="openpyxl") as writer:
    df_com.to_excel(writer, sheet_name="Com Relacionados", index=False)
    df_sem.to_excel(writer, sheet_name="Sem Relacionados", index=False)
    format_sheet(writer.sheets["Com Relacionados"], df_com, COLORS_COM)
    format_sheet(writer.sheets["Sem Relacionados"], df_sem, COLORS_SEM)

print(f"Arquivo '{output_file}' gerado com sucesso.")
print(f"  Aba 'Com Relacionados' : {len(df_com)} linha(s)")
print(f"  Aba 'Sem Relacionados' : {len(df_sem)} linha(s)")

Arquivo '../responses/contatos-e-relacionados/relatorio_contatos_e_relacionados.xlsx' gerado com sucesso.
  Aba 'Com Relacionados' : 270 linha(s)
  Aba 'Sem Relacionados' : 15 linha(s)
